In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [3]:
# 시계열 종가 예측
df = pd.read_csv('../csv/AAPL.csv')

df = df[['Date', 'Adj Close', 'Volume']]
df.dropna(axis= 0, inplace=True)

In [4]:
X_all = df[['Adj Close', 'Volume']].astype('float').values
Y_all = df[['Adj Close']].astype('float').values

In [6]:
# 데이터의 분할 -> 4개의 데이터가 생성 -> 8:2
# train_test_split() 함수와 같은 부분
split_idx = int(len(df) * 0.8)

X_train, X_test = X_all[: split_idx], X_all[split_idx: ]
Y_train, Y_test = Y_all[: split_idx], Y_all[split_idx: ]

In [7]:
# 스케일링
# 스케일링 따로 사용 -> 마지막 예측 종가, 실제 종가 데이터의 원본으로 역변환을 편하게 하기 위해
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_sc = x_scaler.fit_transform(X_train)
X_test_sc = x_scaler.transform(X_test)

Y_train_sc = y_scaler.fit_transform(Y_train)
Y_test_sc = y_scaler.transform(Y_test)

In [ ]:
# DataLoader에서 구간 데이터를 나눠주기 위한 DataSet 구성
class WindowDataset(Dataset):
    def __init__(self, _x, _y, _window):
        # _x : 독립
        # _y : 종속
        # window : 구간
        self.x = _x
        self.y = _y
        self.window = _window
        self.n = len(x) - _window